# 02 — Quality control, spatially

**Day 1, 11:00–12:00**

### Where we are going

Everything you know about scRNA-seq QC applies here, and none of it is sufficient.
The extra questions are:

- **Is the chemistry clean?** One number, `control_frac`, from the negative controls.
- **Is the segmentation sane?** Answered by counts-vs-area, and by looking.
- **Are the problems spread evenly, or concentrated in one part of the tissue?**
  Answered only by plotting QC **in space** — and this is the move people coming from
  scRNA-seq forget.

By the end you can:

1. build per-cell QC metrics that mean something for imaging-based data
2. spot segmentation failures from the counts–area relationship
3. map every QC metric onto the tissue and recognise technical artefacts
4. work out **which** filter is removing which cells, and whether that is defensible

> **Section 4 is yours to decide.** The notebook runs as written if you leave the
> numbers alone, but the point of that section is that you choose your own thresholds
> and then look at what they did to the tissue. There is no correct answer, and we
> will compare across the room afterwards.
>
> Appendix A (`A1_controls_and_detection_limits.ipynb`) is optional extra depth.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=90, frameon=False)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

NAVY, GOLD, CORAL, ICE = "#001158", "#FBAE40", "#F26B43", "#BCD2FF"

In [ ]:
adata = sc.read_h5ad(DATA / "ovarian_subset.h5ad")
adata.layers["counts"] = adata.X.copy()      # keep raw counts safe, always
adata

In [ ]:
def flag_controls(adata, verbose=True):
    """Which features are controls rather than targeted genes?

    `feature_types` from the 10x file is authoritative: anything that is not
    "Gene Expression" is a control of some kind. Name matching is a fallback,
    because control naming has changed between Xenium chemistries — and a
    detector that silently finds nothing is worse than one that finds too much.
    """
    names = adata.var_names.str.lower()
    by_name = (
        names.str.startswith("negcontrol") | names.str.startswith("neg_control")
        | names.str.startswith("unassignedcodeword")
        | names.str.startswith("unassigned_codeword")
        | names.str.startswith("deprecatedcodeword")
        | names.str.startswith("genomiccontrol")
        | names.str.startswith("genomic_control")
        | names.str.startswith("antisense") | names.str.startswith("blank")
        | names.str.contains("codeword")
    )
    by_name = pd.Series(np.asarray(by_name), index=adata.var_names)

    if "feature_types" in adata.var.columns:
        ft = adata.var["feature_types"].astype(str).str.strip().str.lower()
        control = ~ft.isin(["gene expression", "gene_expression"]) | by_name
    else:
        control = by_name

    if verbose:
        n = int(control.sum())
        print(f"{n} control features, {int((~control).sum())} targeted genes")
        if n:
            print("examples:", list(adata.var_names[control.to_numpy()][:5]))
        else:
            print("\n*** No controls found — the rest of section 1 cannot run. ***")
            print("var columns:", list(adata.var.columns))
            if "feature_types" in adata.var.columns:
                print(adata.var["feature_types"].value_counts())
            print("first 10 feature names:", list(adata.var_names[:10]))
            print("\nMost likely cause: the matrix was built with scanpy's")
            print("read_10x_h5(gex_only=True) default, which keeps only")
            print("'Gene Expression' features and discards every control.")
            print("The controls are in the original .h5 but not in this file, so")
            print("no code here can recover them — the data must be regenerated.")
            print("Tell the organiser; section 1 is the only part affected.")
    return control


# Recompute rather than trusting the stored column: older versions of the data
# preparation script flagged controls by name only, and on some panels that
# finds nothing.
adata.var["control"] = flag_controls(adata)

## 1. Per-cell metrics

Five numbers per cell, and one of them has no equivalent in scRNA-seq.

**`control_frac`** is the fraction of a cell's signal landing on features designed to
measure nothing — probes against absent sequences, and codewords no probe uses. It is
a direct read on how much of that cell is noise.

One subtlety, because it bites: the matrix contains several other non-gene feature
classes that are **not** controls. Deprecated codewords in particular are retired
probes that still detect real transcripts, and they can carry hundreds of thousands of
counts. Including them would turn `control_frac` into a measure of biology. The cell
below uses only the two true negative-control classes.

> Appendix A (`A1_controls_and_detection_limits.ipynb`) covers the rest: the control
> classes in full, and how to derive a gene-level detection threshold from them. It
> is optional and nothing here depends on it.

In [ ]:
def flag_controls(adata, verbose=True):
    """Split the matrix into targeted genes and everything else.

    `feature_types` from the 10x file is authoritative: anything that is not
    "Gene Expression" is a non-gene feature. Only two of those classes are
    designed as negative controls, and only those two measure background —
    see Appendix A if you want the detail.
    """
    ft = (adata.var["feature_types"].astype(str)
          if "feature_types" in adata.var.columns
          else pd.Series("unknown", index=adata.var_names))
    is_gene = ft.str.strip().str.lower().isin(["gene expression", "gene_expression"])

    names = adata.var_names.str.lower()
    looks_control = (names.str.startswith(("negcontrol", "unassignedcodeword",
                                           "deprecatedcodeword", "genomiccontrol",
                                           "antisense", "blank"))
                     | names.str.contains("codeword"))
    is_ctrl = (~is_gene) | pd.Series(np.asarray(looks_control), index=adata.var_names)

    BACKGROUND = ["Negative Control Probe", "Negative Control Codeword"]
    is_background = ft.isin(BACKGROUND) & is_ctrl
    if not is_background.any():
        if verbose:
            print("No designated negative controls found; using all non-gene features.")
        is_background = is_ctrl

    if verbose:
        print(f"{int((~is_ctrl).sum()):>6,} targeted genes")
        print(f"{int(is_background.sum()):>6,} negative controls   <- background comes from these")
        print(f"{int((is_ctrl & ~is_background).sum()):>6,} other non-gene features (deprecated "
              "codewords etc.) — NOT background")
        if int((~is_ctrl).sum()) == 0:
            print("\n*** No genes found. Was the matrix loaded with gex_only=True? ***")
    return is_ctrl.to_numpy(), is_background.to_numpy()


is_ctrl, is_background = flag_controls(adata)

In [ ]:
# Per-cell background: only the true negative controls. Summing every non-gene
# feature instead would include deprecated codewords, which are retired probes
# still detecting real transcripts — that turns "background" into a measure of
# biology, and any threshold on it removes cells for expressing the wrong genes.
adata.obs["negctrl_counts"] = np.asarray(
    adata.layers["counts"][:, is_background].sum(axis=1)).ravel()

# keep only real genes from here on
adata = adata[:, ~is_ctrl].copy()
sc.pp.calculate_qc_metrics(adata, percent_top=None, inplace=True, log1p=False)

empty = (adata.obs["total_counts"] == 0).to_numpy()
if empty.any():
    print(f"{empty.sum():,} cells ({100 * empty.mean():.2f}%) have zero gene counts "
          "after removing non-gene features.\n")

denom = adata.obs["total_counts"] + adata.obs["negctrl_counts"]
adata.obs["control_frac"] = np.where(
    denom > 0, adata.obs["negctrl_counts"] / denom.replace(0, np.nan), 0.0)
adata.obs["nucleus_ratio"] = adata.obs["nucleus_area"] / adata.obs["cell_area"].replace(0, np.nan)
adata.obs["counts_per_um2"] = adata.obs["total_counts"] / adata.obs["cell_area"].replace(0, np.nan)

print(f"median control_frac: {adata.obs['control_frac'].median():.6f}")
print("A clean Xenium run sits around 1e-4. Much above 1e-2 and the weak genes in")
print("your panel are not trustworthy.\n")

adata.obs[["total_counts", "n_genes_by_counts", "cell_area", "nucleus_ratio",
           "control_frac", "counts_per_um2"]].describe().round(4)

In [ ]:
metrics = [
    ("total_counts", "transcripts per cell", True),
    ("n_genes_by_counts", "genes detected per cell", False),
    ("cell_area", "cell area (µm²)", True),
    ("nucleus_ratio", "nucleus / cell area", False),
    ("control_frac", "negative-control fraction", False),
    ("counts_per_um2", "transcript density (per µm²)", False),
]

fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax in axes.ravel()[len(metrics):]:
    ax.axis("off")
for ax, (col, label, logx) in zip(axes.ravel(), metrics):
    v = adata.obs[col].to_numpy()
    v = v[np.isfinite(v)]
    if logx:
        # Bin in log space. Plotting linear bins on a log axis gives bars of
        # wildly uneven width and squashes the left-hand tail into a wall.
        v = v[v > 0]
        bins = np.logspace(np.log10(v.min()), np.log10(v.max()), 60)
        ax.set_xscale("log")
    else:
        bins = 60
    ax.hist(v, bins=bins, color=NAVY)
    ax.set_xlabel(label); ax.set_ylabel("cells")
    ax.axvline(np.median(v), color=GOLD, lw=2)
sns.despine(); fig.suptitle("Per-cell QC (gold line = median)", y=1.02)
fig.tight_layout(); plt.show()

### Read the numbers out loud

**Median transcripts per cell.** Compare it with a 10x 3' scRNA-seq experiment
(5,000–20,000 UMIs). You have one to two orders of magnitude less. Consequences:

- per-cell expression of any single gene is close to binary — a cell has 0, 1 or 2
  copies of most transcripts
- gene–gene correlation within a cell is nearly meaningless
- but you have *many* cells and you know where they are, so **aggregate across space,
  not within cells**

That reframing — from "what does this cell express" to "what does this region
express" — is the single biggest mental shift for someone arriving from scRNA-seq.

**Nucleus/cell ratio near 1.0** means the segmentation gave that cell essentially no
cytoplasm. A pile-up at exactly 1.0 suggests nucleus-expansion segmentation rather
than true boundary detection.

## 2. Counts versus area — where segmentation failures show up

In a healthy dataset, bigger cells have more transcripts, roughly proportionally.
Deviations tell you specific stories.

In [ ]:
# Log axes cannot show zeros, and some cells genuinely have none. Drop them from
# THIS PLOT only, and report how many — a cell with zero gene counts is a QC
# finding, not just a plotting nuisance.
area = adata.obs["cell_area"].to_numpy()
counts = adata.obs["total_counts"].to_numpy()
ok = (area > 0) & (counts > 0)

n_zero_counts = int((counts <= 0).sum())
n_zero_area = int((area <= 0).sum())
if n_zero_counts:
    print(f"{n_zero_counts:,} cells ({100 * n_zero_counts / adata.n_obs:.2f}%) have zero "
          f"gene counts and cannot be shown on a log axis.")
    print("These are usually polygons whose only counts were on control features,")
    print("or empty segmentations. They will be removed by the filter in section 5.")
if n_zero_area:
    print(f"{n_zero_area:,} cells have zero area — check the segmentation output.")

# hexbin must be told to bin in log space itself. Calling ax.set_xscale("log")
# afterwards bins linearly and then stretches the hexagons, which smears
# everything into one dark rectangle.
fig, ax = plt.subplots(figsize=(6.5, 5.5))
h = ax.hexbin(area[ok], counts[ok],
              xscale="log", yscale="log",          # <- the important part
              gridsize=70, bins="log", cmap="viridis", mincnt=1)
ax.set_xlabel("cell area (µm²)"); ax.set_ylabel("transcripts per cell")
plt.colorbar(h, ax=ax, label="log10(cells per bin)")
ax.set_title(f"counts vs area  ({int(ok.sum()):,} cells shown)")

# median counts per area decile, to show the trend through the cloud
sub = adata.obs.loc[ok]
q = pd.qcut(sub["cell_area"], 12, labels=False, duplicates="drop")
trend = sub.groupby(q, observed=True).agg(
    area=("cell_area", "median"), counts=("total_counts", "median"))
ax.plot(trend["area"], trend["counts"], color=CORAL, lw=2.2,
        marker="o", ms=4, label="median per area decile")
ax.legend(loc="lower right", framealpha=0.9)
plt.show()

r = np.corrcoef(np.log10(area[ok]), np.log10(counts[ok]))[0, 1]
print(f"\ncorrelation of log(area) with log(counts): r = {r:.2f}")

Four regions of this plot, and what each means:

| Where | Likely cause |
|---|---|
| bottom-left (small + empty) | debris, cut nuclei at the section edge, failed segmentation |
| not shown at all | cells with **zero** gene counts — a log axis cannot display them, so the cell above counts them for you |
| top-right (large + rich) | **merged cells** — two cells in one polygon (the spatial doublet) |
| bottom-right (large + empty) | polygon drawn over extracellular space, or a fat adipocyte-like cell |
| main diagonal | ordinary cells |

Note that "spatial doublet" is *not* the same as a scRNA-seq doublet, and the tools
are different: `scrublet`/`DoubletFinder` model two transcriptomes summed in a
droplet. Here you have a geometrical failure, and the fix is geometric — look at the
polygon, or re-segment. **Do not run scRNA-seq doublet detection on Xenium data and
believe the output.**

The orange line is the median counts per area decile. In healthy data it climbs
steadily; a plateau at the right-hand end means the largest polygons are *not*
gaining transcripts in proportion, which is what you would expect if they are
enclosing empty space rather than more cell.


## 3. The step people skip: put QC on the tissue

If your low-count cells sit in one stripe, that is an instrument or focus problem.
If they sit in a coherent tissue region, they may be real biology — necrosis,
stroma, a dense lymphoid aggregate — and filtering them silently deletes a
structure from your paper.

In [ ]:
def spatial_plot(adata, color, ax=None, s=1.2, cmap="viridis", vmin=None, vmax=None,
                 title=None, categorical=False, legend=False):
    """Minimal spatial scatter. Deliberately hand-rolled so you can see the mechanics:
    it is just obsm['spatial'] with a colour."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5.5, 5.5))
    x, y = adata.obsm["spatial"].T
    if categorical:
        cats = adata.obs[color].astype("category")
        codes = cats.cat.codes
        cm = plt.get_cmap("tab20")
        ax.scatter(x, y, s=s, c=[cm(i % 20) for i in codes], linewidths=0, rasterized=True)
        if legend:
            for i, name in enumerate(cats.cat.categories):
                ax.scatter([], [], s=25, color=cm(i % 20), label=name)
            ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=8)
    else:
        v = adata.obs[color].to_numpy() if color in adata.obs else np.asarray(
            adata[:, color].X.todense()).ravel()
        pc = ax.scatter(x, y, s=s, c=v, cmap=cmap, vmin=vmin, vmax=vmax,
                        linewidths=0, rasterized=True)
        plt.colorbar(pc, ax=ax, fraction=0.046, label=color)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title or color)
    return ax

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.2))
spatial_plot(adata, "total_counts", ax=axes[0], vmax=np.percentile(adata.obs["total_counts"], 98),
             title="transcripts per cell")
spatial_plot(adata, "cell_area", ax=axes[1], vmax=np.percentile(adata.obs["cell_area"], 98),
             title="cell area")
spatial_plot(adata, "control_frac", ax=axes[2], vmax=np.percentile(adata.obs["control_frac"], 98),
             cmap="magma", title="control fraction")
plt.tight_layout(); plt.show()

> **Try it yourself — colour the tissue by anything**
>
> The three maps above use three QC columns. Every column in `adata.obs` can be
> plotted this way. Pick a different one and see whether it has spatial structure.

In [ ]:
# What can you colour by? These are the numeric columns available.
numeric_cols = [c for c in adata.obs.columns
                if pd.api.types.is_numeric_dtype(adata.obs[c])]
print(numeric_cols)

In [ ]:
COLOUR_BY = "n_genes_by_counts"      # <-- CHANGE THIS to any name printed above

spatial_plot(adata, COLOUR_BY,
             vmax=np.percentile(adata.obs[COLOUR_BY].dropna(), 98),
             title=COLOUR_BY)
plt.show()

print(adata.obs[COLOUR_BY].describe().round(2))

### What to look for, concretely

- **Straight edges, stripes or a grid.** Xenium images in fields of view (FOVs) and
  stitches them. A visible tiling pattern is technical, full stop.
- **A gradient across the section.** Often focus, tissue thickness, or permeabilisation.
- **A blob of low counts with a soft boundary.** Usually biological — necrotic core,
  dense collagen, adipose.
- **Anything that follows a fold or a tear.** Section damage; exclude the region
  explicitly rather than letting a count threshold do it invisibly.

### Exercise 2.2
Is there a region of this crop you would exclude entirely, before any per-cell
filtering? Draw its bounding box, count how many cells it holds, and justify it in
one sentence you would be willing to put in a methods section.

In [ ]:
# your code here — a rectangle mask on adata.obsm["spatial"] is enough

## 4. Choose your own filters

**This is the part you do yourself.** There is no correct set of thresholds for a
Xenium section, and anyone who tells you otherwise is quoting a paper that used a
different tissue.

The workflow, and it is the same one you will use on your own data:

1. **Look** at the distributions and propose a number.
2. **Apply** it and see how many cells you lose.
3. **Plot what you removed, in space.** This is the step that matters.
4. **Decide** — and write the sentence you would put in a methods section.

Work in pairs if you like. In about fifteen minutes we will compare what the room
chose, and the interesting result is usually the spread rather than any one answer.

> Day 2 loads an annotation prepared in advance, so whatever you choose here, you will
> not be stranded tomorrow. Be bold.

In [ ]:
# Step 1 — look before you choose. Where would YOU cut each of these?
fig, axes = plt.subplots(1, 4, figsize=(17, 3.6))
for ax, (col, label, logx) in zip(axes, [
        ("total_counts", "transcripts per cell", True),
        ("n_genes_by_counts", "genes detected", False),
        ("cell_area", "cell area (µm²)", True),
        ("control_frac", "control fraction", False)]):
    v = adata.obs[col].to_numpy()
    v = v[np.isfinite(v)]
    if logx:
        v = v[v > 0]
        bins = np.logspace(np.log10(v.min()), np.log10(v.max()), 60)
        ax.set_xscale("log")
    else:
        bins = 60
    ax.hist(v, bins=bins, color=NAVY)
    for q, c in [(1, CORAL), (5, GOLD), (10, "0.5")]:
        ax.axvline(np.percentile(v, q), color=c, lw=1.6, label=f"{q}th pct")
    ax.set_xlabel(label); ax.set_ylabel("cells")
axes[0].legend(fontsize=7)
sns.despine(); fig.suptitle("Percentile markers are a hint, not an answer", y=1.04)
plt.tight_layout(); plt.show()

for col in ["total_counts", "n_genes_by_counts", "cell_area", "control_frac"]:
    q = adata.obs[col].quantile([0.01, 0.05, 0.10, 0.50, 0.95, 0.99]).round(3)
    print(f"{col:<20} " + "  ".join(f"p{int(100*k)}={v}" for k, v in q.items()))

### Step 2 — choose

Set your four numbers below. Some things worth knowing before you do:

- **`total_counts`** — most people land between 5 and 30. Lower keeps small cells
  (lymphocytes, dense tumour) at the cost of noisier profiles.
- **`n_genes_by_counts`** — usually tracks counts; its job is catching empty polygons.
- **`cell_area`** — a percentile pair rather than absolute values, since area
  distributions differ between tissues.
- **`control_frac`** — on a clean run the median is around 1e-4, so almost any value
  above 0.01 does nothing. Set it deliberately rather than copying 0.05.

There is no penalty for being wrong. You are about to see exactly what your choice
did.

In [ ]:
# ---- YOUR CHOICE ---------------------------------------------------------
MIN_COUNTS = 10           # <-- CHANGE THIS
MIN_GENES = 5             # <-- CHANGE THIS
MAX_CONTROL_FRAC = 0.05   # <-- CHANGE THIS
AREA_PERCENTILES = (1, 99)   # <-- CHANGE THIS (lower, upper)
# --------------------------------------------------------------------------

area_lo, area_hi = np.percentile(adata.obs["cell_area"].dropna(), AREA_PERCENTILES)

reasons = {
    "low counts": (adata.obs["total_counts"] < MIN_COUNTS).to_numpy(),
    "few genes": (adata.obs["n_genes_by_counts"] < MIN_GENES).to_numpy(),
    "high control frac": (adata.obs["control_frac"] > MAX_CONTROL_FRAC).to_numpy(),
    "area too small": (adata.obs["cell_area"] < area_lo).to_numpy(),
    "area too large": (adata.obs["cell_area"] > area_hi).to_numpy(),
}
keep = ~np.logical_or.reduce(list(reasons.values()))
adata.obs["qc_pass"] = keep

# record the choice so it travels with the object and ends up in your methods
adata.uns["qc_thresholds"] = {
    "min_counts": MIN_COUNTS, "min_genes": MIN_GENES,
    "max_control_frac": MAX_CONTROL_FRAC,
    "area_percentiles": list(AREA_PERCENTILES),
    "area_bounds_um2": [round(float(area_lo), 1), round(float(area_hi), 1)],
    "n_kept": int(keep.sum()), "n_total": int(adata.n_obs),
}

print(f"keeping {keep.sum():,} of {adata.n_obs:,} cells ({100 * keep.mean():.1f}%)\n")
for name, m in reasons.items():
    print(f"  removed by {name:<20} {int(m.sum()):>7,}")
print(f"\n  area bounds: {area_lo:.1f} to {area_hi:.1f} um2")

if keep.mean() < 0.5:
    print("\n*** You are discarding more than half the section. That is a strong")
    print("    claim about the data. Make sure step 3 justifies it. ***")
elif keep.mean() > 0.99:
    print("\nYour filter removes almost nothing. That is a legitimate choice on a")
    print("clean run — but check step 3 to be sure it is not hiding a bad region.")

### Step 3 — look at what you removed

Not at how many. **Where.**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
x, y = adata.obsm["spatial"].T
axes[0].scatter(x[keep], y[keep], s=1, c=ICE, linewidths=0, rasterized=True)
axes[0].scatter(x[~keep], y[~keep], s=1.6, c=CORAL, linewidths=0, rasterized=True)
axes[0].set_title("kept (blue) vs discarded (orange)")

axes[1].hexbin(x[~keep], y[~keep], gridsize=45, cmap="Oranges", mincnt=1)
axes[1].set_title("density of discarded cells")
for ax in axes:
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

### Step 4 — decide, and write it down

Look at the maps above and answer three questions.

1. **Do the discarded cells form a structure?** If yes, your filter is removing a
   population, not noise.
2. **Which filter did it?** The panels below split it out. "Low counts over a tumour
   nest" and "high control fraction over a fold" call for completely different
   responses.
3. **Can you defend it in one sentence?** Write it now, in the cell below. If you
   cannot write the sentence, change the threshold.

Then compare with your neighbour. If you kept 92% and they kept 78%, one of you is
not wrong — but you should be able to say what the difference costs.

In [ ]:
# Write your methods sentence here. Be specific: numbers, and what you lost.
#
# Example of the shape it should take:
#   "Cells with fewer than 10 transcripts, fewer than 5 detected genes, or a cell
#    area outside the 1st-99th percentile were excluded (n = 4,812; 12.3%). Removed
#    cells were enriched in densely packed tumour nests, so tumour cell frequencies
#    are underestimated."

MY_METHODS_SENTENCE = """
    ...
"""

adata.uns["qc_thresholds"]["methods_sentence"] = MY_METHODS_SENTENCE.strip()
print(adata.uns["qc_thresholds"]["methods_sentence"])
print(f"\n(recorded alongside your thresholds: {adata.uns['qc_thresholds']['n_kept']:,} "
      f"of {adata.uns['qc_thresholds']['n_total']:,} cells kept)")

### Which filter is doing it?

"The discarded cells form a structure" is the finding. **Which filter caught them** is
the actionable part, because the four filters fail for entirely different reasons.

The next cell maps each one separately.

In [ ]:
# `reasons` was built with YOUR thresholds in step 2 — reuse it.
fig, axes = plt.subplots(1, len(reasons), figsize=(4.0 * len(reasons), 4.3))
for ax, (name, m) in zip(np.atleast_1d(axes), reasons.items()):
    ax.scatter(x, y, s=0.5, c="0.9", linewidths=0, rasterized=True)
    ax.scatter(x[m], y[m], s=1.6, c=CORAL, linewidths=0, rasterized=True)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{name}\n{m.sum():,} cells ({100 * m.mean():.1f}%)", fontsize=10)
plt.tight_layout(); plt.show()

overlap = pd.DataFrame({
    "removed by this": {k: int(v.sum()) for k, v in reasons.items()},
    "ONLY this one": {k: int((v & ~np.logical_or.reduce(
        [w for j, w in reasons.items() if j != k])).sum()) for k, v in reasons.items()},
})
display(overlap)

### Reading the panels

Whichever panel reproduces the structure you saw is the one to think about.

**Low counts / few genes over a tumour nest.** The usual cause is not bad data but
**dense packing**. Tumour cells in a nest are tightly apposed with little cytoplasm,
so segmentation gives small polygons that capture few transcripts. The measurement is
working; the cells are genuinely small compartments.

A second contributor in very dense, highly expressing regions is **optical crowding**:
transcripts sitting so close that the decoder cannot resolve them, so they fail the
decoder's own confidence threshold and never reach the matrix. That shows up as a *relative* drop in counts
exactly where expression is highest, which is counter-intuitive and real.

**Area too small over a nest.** Same mechanism, seen through the other filter.

**Area too large.** Merged polygons — two or more cells in one boundary. Expect this
at the *edges* of dense regions rather than in their cores.

**High control fraction over a region.** Different in kind. This one is genuinely
about signal quality: those cells have proportionally more background. If it maps to a
tissue fold or a section edge, exclude the region explicitly rather than by threshold.

**Necrosis** is the other real possibility for a tumour core: degraded RNA, genuinely
low counts, and a biologically meaningful region you probably want to keep and label
rather than silently delete.

In [ ]:
# Compare REGIONS, not kept-vs-discarded. Comparing discarded cells against kept
# ones on a count-derived metric is circular: the filter selected them on counts,
# so of course they look sparse. Instead, find the tiles where discarding is
# concentrated and compare EVERY cell there against every cell elsewhere.
BIN = 75.0        # um; big enough that each tile holds a few dozen cells

gx = ((x - x.min()) // BIN).astype(int)
gy = ((y - y.min()) // BIN).astype(int)
tile = pd.Series(list(zip(gx, gy)), index=adata.obs_names)

disc = ~keep
rate = (pd.DataFrame({"tile": tile.to_numpy(), "disc": disc})
        .groupby("tile")["disc"].agg(["mean", "size"]))
rate = rate[rate["size"] >= 20]
hot = set(rate[rate["mean"] >= rate["mean"].quantile(0.90)].index)
in_hot = tile.isin(hot).to_numpy()

print(f"high-discard tiles hold {in_hot.sum():,} cells; discard rate there "
      f"{100 * disc[in_hot].mean():.1f}% vs {100 * disc[~in_hot].mean():.1f}% elsewhere\n")

cols = ["cell_area", "counts_per_um2", "total_counts", "nucleus_ratio", "control_frac"]
cmp = pd.DataFrame({
    "high-discard region": adata.obs.loc[in_hot, cols].median(),
    "rest of section": adata.obs.loc[~in_hot, cols].median(),
}).round(3)
cmp["ratio"] = (cmp["high-discard region"] / cmp["rest of section"]).round(2)
display(cmp)

area_r = cmp.loc["cell_area", "ratio"]
dens_r = cmp.loc["counts_per_um2", "ratio"]
cf_r = cmp.loc["control_frac", "ratio"]
print("Interpretation:")
if area_r < 0.8 and dens_r > 0.8:
    print("  Cells there are SMALLER but just as transcript-dense. The tissue is")
    print("  fine; your filter is removing cells for being small. Packing and")
    print("  segmentation, not data quality.")
elif dens_r < 0.8 and cf_r < 1.5:
    print("  The tissue there yields less per unit area. Think necrosis, poor")
    print("  fixation, or a fold — a real regional difference in the sample.")
elif cf_r > 1.5:
    print("  Negative-control background is proportionally higher there. This one IS")
    print("  a signal-quality problem; consider excluding the region explicitly.")
else:
    print("  No clear geometric explanation — look at the region in Xenium Explorer.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
axes[0].scatter(x, y, s=0.5, c="0.88", linewidths=0, rasterized=True)
axes[0].scatter(x[in_hot], y[in_hot], s=1.4, c=NAVY, linewidths=0, rasterized=True)
axes[0].set_title("the region being compared")

pc = axes[1].scatter(x, y, c=adata.obs["cell_area"], s=1.0, cmap="viridis",
                     vmax=np.percentile(adata.obs["cell_area"].dropna(), 98),
                     linewidths=0, rasterized=True)
plt.colorbar(pc, ax=axes[1], label="cell area (µm²)")
axes[1].set_title("cell area — does it match?")
for ax in axes:
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

### If `control_frac` flagged the region

A high negative-control fraction is a real quality problem: those cells genuinely have
proportionally more measurable noise. If it maps onto a fold or a section edge, exclude
that region explicitly rather than by threshold.

One warning, because it is easy to get wrong. `control_frac` here counts **only the
two true negative-control classes**. If you compute it over every non-gene feature — as
is tempting, and as several published pipelines do — you sweep in deprecated codewords,
which are retired probes still detecting real transcripts. That version of the metric
runs about four orders of magnitude higher and tracks biology, so thresholding on it
removes whole cell types. Appendix A works through the arithmetic.

### So what do you actually do?

Four defensible options. They are not equally good, and the choice depends on your
question.

**1. Keep the threshold, name the casualty.** Perfectly acceptable *if you say so*:
"cells with <10 transcripts were excluded; these were enriched in densely packed
tumour nests, so tumour cell frequencies are underestimated." A reader can then
discount your composition estimates appropriately.

**2. Lower the threshold and accept noisier cells.** Reasonable if you care about
*where* tumour cells are rather than what each one expresses. Position survives low
counts far better than expression does.

**3. Threshold on density, not on counts.** `counts_per_um2` asks whether a cell
yielded a reasonable amount *for its size*, which does not penalise small cells.
Try it as an alternative filter and see whether the structure disappears.

**4. Filter within region.** Compute thresholds separately inside and outside dense
regions. Most defensible, most work, and hardest to explain in a methods section.

**What is not acceptable** is option 1 without the second half of the sentence.

> **The general lesson.** Every QC threshold is a hypothesis that low-quality cells
> are randomly distributed. Spatial data lets you test that hypothesis, and here it
> failed. In scRNA-seq the same thing happens — dense, small, fragile cells are lost
> at dissociation and at filtering — and you simply cannot see it.

### Exercise 2.3b
Re-filter using `counts_per_um2` instead of `total_counts`, at a threshold that
removes a similar total number of cells. Does the spatial structure in the discarded
set weaken? If it does, your original filter was measuring cell size more than data
quality.

In [ ]:
# your code here

In [ ]:
t = adata.uns["qc_thresholds"]
print("Your choice, in one line — read this out when we compare:\n")
print(f"  counts>={t['min_counts']}  genes>={t['min_genes']}  "
      f"control_frac<={t['max_control_frac']}  "
      f"area p{t['area_percentiles'][0]}-p{t['area_percentiles'][1]}"
      f"   ->  kept {t['n_kept']:,} / {t['n_total']:,} "
      f"({100 * t['n_kept'] / t['n_total']:.1f}%)")

In [ ]:
adata = adata[keep].copy()
adata.write_h5ad(DATA / "ovarian_qc.h5ad", compression="gzip")
print(adata)
print(f"\nsaved with your thresholds recorded in adata.uns['qc_thresholds']")
print("Notebook 03 picks up from this file, so it inherits your choice.")

### Exercise 2.3 — compare with someone else

Get your neighbour's four numbers and run them alongside yours.

1. How many cells does each choice keep?
2. Plot the cells that **one of you keeps and the other discards**. Where are they?
3. Which cell types are they likely to be? You do not have labels yet — use position
   and cell area as proxies.

The point is not to agree. It is to be able to say what the disagreement costs, in
cells and in tissue.

### Exercise 2.4 — the honest version
Take your thresholds and make them deliberately too strict — say `MIN_COUNTS = 50`.
Run the whole section again. Now write the methods sentence for *that* filter. If you
can write a sentence you would be comfortable defending in review, your original
threshold may have been too cautious.